# All-22 Pre-Annotation — GPU-Accelerated

Runs the fine-tuned YOLOv12x model on extracted training frames to generate
pre-annotations for correction in Label Studio.

**Setup before running:**
1. Upload new extracted frames to `all22_training/images/` in Google Drive (same folder as before — new filenames won't collide)
2. Make sure `best.pt` is at `all22_training/checkpoints/best.pt` in Google Drive
3. Switch runtime to **T4 GPU** (Runtime → Change runtime type)

**Output:** YOLO `.txt` label files saved to `all22_training/new_labels/` in Drive.
Your existing hand-done labels in `export_final/labels/` are never touched.

## Step 1 — Mount Drive and verify files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR    = '/content/drive/MyDrive/all22_training'
IMAGES_DIR   = f'{DRIVE_DIR}/images'       # shared folder with all frames (old + new)
LABELS_DIR   = f'{DRIVE_DIR}/new_labels'   # pre-annotations for new frames only
MODEL_PATH   = f'{DRIVE_DIR}/checkpoints/best.pt'

os.makedirs(LABELS_DIR, exist_ok=True)

# Only process images that don't already have hand-done labels
# Hand-done labels live in export_final/labels/ — check against those too
HAND_DONE_DIR = f'{DRIVE_DIR}/export_final/labels'
hand_done = set(os.listdir(HAND_DONE_DIR)) if os.path.exists(HAND_DONE_DIR) else set()

all_images = sorted([f for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg')])
# Skip frames that already have hand-done annotations
new_images = [f for f in all_images if f.replace('.jpg', '.txt') not in hand_done]

print(f'Total images in folder: {len(all_images)}')
print(f'Already hand-labeled:   {len(all_images) - len(new_images)} (will skip)')
print(f'To pre-annotate:        {len(new_images)}')
print(f'Model exists:           {os.path.exists(MODEL_PATH)}')

import torch
print(f'GPU available: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime!"}')

## Step 2 — Install dependencies and load model

In [ ]:
!pip install ultralytics==8.4.31 -q

import shutil
from ultralytics import YOLO
import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

import torch
print(f'Torch version: {torch.__version__}')

# Copy model locally for faster access
shutil.copy(MODEL_PATH, '/content/best.pt')
model = YOLO('/content/best.pt')
print('Model loaded.')

## Step 3 — Run pre-annotation

Adjust `CONF_THRESH` if needed. 0.60 filters out most false positives while
catching the majority of real players. Lower = more boxes to delete, higher = more
boxes to draw.

In [ ]:
import cv2

CONF_THRESH = 0.60
IMGSZ       = 1280

already_done = set(os.listdir(LABELS_DIR))
to_process   = [f for f in new_images if f.replace('.jpg', '.txt') not in already_done]
print(f'To process: {len(to_process)} (skipping {len(new_images) - len(to_process)} already done)')

total_detections = 0
for i, fname in enumerate(to_process):
    img_path = f'{IMAGES_DIR}/{fname}'
    frame    = cv2.imread(img_path)
    h, w     = frame.shape[:2]

    results = model(frame, imgsz=IMGSZ, device='cuda', verbose=False)[0]
    boxes   = results.boxes
    mask    = boxes.conf >= CONF_THRESH
    boxes   = boxes[mask]

    label_path  = f'{LABELS_DIR}/{fname.replace(".jpg", ".txt")}'

    n = 0
    with open(label_path, 'w') as f:
        for box in boxes.xywhn.cpu().numpy():
            cx, cy, bw, bh = box
            f.write(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
            n += 1

    total_detections += n

    if (i + 1) % 50 == 0 or (i + 1) == len(to_process):
        avg = total_detections / (i + 1)
        print(f'  [{i+1}/{len(to_process)}] {avg:.1f} detections/frame avg')

print(f'\nDone! {total_detections} total detections across {len(to_process)} frames')
print(f'Labels saved to Drive: {LABELS_DIR}')

## Step 4 — Sanity check

Draws boxes on a random sample of frames so you can visually verify quality
before downloading everything.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

sample = random.sample(to_process, min(6, len(to_process)))
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for ax, fname in zip(axes, sample):
    img   = cv2.imread(f'{IMAGES_DIR}/{fname}')
    img   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w  = img.shape[:2]
    label = f'{LABELS_DIR}/{fname.replace(".jpg", ".txt")}'
    n     = 0
    if os.path.exists(label):
        with open(label) as f:
            for line in f:
                _, cx, cy, bw, bh = map(float, line.strip().split())
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
                n += 1
    ax.imshow(img)
    ax.set_title(f'{fname[:30]}...\n{n} detections', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 5 — Generate Label Studio import JSON

Creates the import file for Label Studio with pre-annotations loaded.
Download `label_studio_import.json` from Drive and import it into your project.

In [ ]:
import json

# Build import for ALL images (old hand-done + new pre-annotated)
# Label Studio will show hand-done annotations for old frames, predictions for new ones
tasks = []
for fname in all_images:
    # Check hand-done labels first, then pre-annotated
    hand_label = f'{HAND_DONE_DIR}/{fname.replace(".jpg", ".txt")}'
    pre_label  = f'{LABELS_DIR}/{fname.replace(".jpg", ".txt")}'
    label_path = hand_label if os.path.exists(hand_label) else pre_label

    task = {
        'data': {'image': f'/data/local-files/?d=images/{fname}'},
        'predictions': [{'model_version': 'yolo12x_finetuned_60ep', 'result': []}]
    }
    if os.path.exists(label_path):
        with open(label_path) as f:
            for i, line in enumerate(f):
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                _, cx, cy, bw, bh = map(float, parts)
                task['predictions'][0]['result'].append({
                    'id': f'box_{i}',
                    'type': 'rectanglelabels',
                    'from_name': 'label',
                    'to_name': 'image',
                    'value': {
                        'rectanglelabels': ['player'],
                        'x': (cx - bw/2) * 100,
                        'y': (cy - bh/2) * 100,
                        'width': bw * 100,
                        'height': bh * 100,
                        'rotation': 0,
                    }
                })
    tasks.append(task)

out_path = f'{DRIVE_DIR}/label_studio_import.json'
with open(out_path, 'w') as f:
    json.dump(tasks, f, indent=2)

print(f'Label Studio import: {out_path}')
print(f'{len(tasks)} total tasks ({len(all_images) - len(new_images)} hand-labeled + {len(new_images)} pre-annotated)')